In [3]:
%pip install statsmodels

   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.6 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.6 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.6 MB 1.1 MB/s eta 0:00:09
   --- ------------------------------------ 0.8/9.6 MB 1.0 MB/s eta 0:00:09
   ---- ----------------------------------- 1.0/9.6 MB 946.6 kB/s eta 0:00:10
   ---- ----------------------------------- 1.0/9.6 MB 946.6 kB/s eta 0:00:10
   ----- ---------------------------------- 1.3/9.6 MB 957.2 kB/s eta 0:00:09
   ------ --------------------------------- 1.6/9.6 MB 982.7 kB/s eta 0:00:09
   ------- -------------------------------- 1.8/9.6 MB 1.0 MB/s eta 0:00:08
   -------- ------------------------------- 2.1/9.6 MB 1.1 MB/s eta 0:00:07
   --------- --------------------------


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
%pip install prophet pmdarima scikit-learn

  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
   ---------------------------------------- 0.0/12.1 MB ? eta -:--:--
    --------------------------------------- 0.3/12.1 MB ? eta -:--:--
   -- ------------------------------------- 0.8/12.1 MB 2.2 MB/s eta 0:00:06
   ---- ----------------------------------- 1.3/12.1 MB 2.5 MB/s eta 0:00:05
   ------ --------------------------------- 1.8/12.1 MB 2.5 MB/s eta 0:00:05
   ------- -------------------------------- 2.4/12.1 MB 2.5 MB/s eta 0:00:04
   --------- ------------------------------ 2.9/12.1 MB 2.6 MB/s eta 0:00:04
   ----------- ---------------------------- 3.4/12.1 MB 2.5 MB/s eta 0:00:04
   ------------ --------------------------- 3.9/12.1 MB 2.5 MB/s eta 0:00:04
   --------------- ------------------------ 4.7/12.1 MB 2.6 MB/s eta 0:00:03
   ----------------- ---------------------- 5.2/12.1 MB 2.7 MB/s eta 0:00:03
   ------------------- -------------------- 6.0/12.1 MB 2.7 MB/s eta 0:00:03
   -------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import yfinance as yf
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller
import warnings
warnings.filterwarnings('ignore')

# 1. Fetch Data (Jan 2021 to May 13, 2026)
tickers = ["RELIANCE.NS", "HDFCBANK.NS", "INFY.NS", "TMCV.NS", "ITC.NS"]
print("Fetching historical data up to May 13, 2026...")
data = yf.download(tickers, start="2021-01-01", end="2026-05-14", interval="1d")['Close']

# 2. Handle Missing Values (FIXED: Using bfill instead of dropna to save historical rows)
data_cleaned = data.ffill().bfill()

# 3. Augmented Dickey-Fuller (ADF) Test
print("\n--- ADF Stationarity Test Results ---")
for stock in tickers:
    series = data_cleaned[stock]
    p_value = adfuller(series)[1]
    status = "Stationary" if p_value < 0.05 else "Non-Stationary (Needs Differencing)"
    print(f"{stock.split('.')[0]}: p-value = {p_value:.4f} ({status})")

# 4. Train/Test Split (Last 6 Months: Nov 13, 2025 - May 13, 2026)
split_date = "2025-11-13" 
train_data = data_cleaned[:split_date]
test_data = data_cleaned[split_date:]

print(f"\nTraining Set Shape: {train_data.shape}")
print(f"Testing Set Shape: {test_data.shape}")

[*********************100%***********************]  5 of 5 completed

Fetching historical data up to May 13, 2026...

--- ADF Stationarity Test Results ---
RELIANCE: p-value = 0.2093 (Non-Stationary (Needs Differencing))
HDFCBANK: p-value = 0.3484 (Non-Stationary (Needs Differencing))
INFY: p-value = 0.2041 (Non-Stationary (Needs Differencing))
TMCV: p-value = 0.6021 (Non-Stationary (Needs Differencing))
ITC: p-value = 0.4894 (Non-Stationary (Needs Differencing))

Training Set Shape: (1203, 5)
Testing Set Shape: (123, 5)


In [10]:
from prophet import Prophet
from pmdarima import auto_arima
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

forecast_deliverables = {}

print("Executing Task 3: Model Training on FULL Data & Forecasting...\n")

for stock in tickers:
    stock_name = stock.split('.')[0]
    series = data_cleaned[stock]  # Using the FULL dataset up to May 13
    
    # --- MODEL 1: PROPHET (Trained on Full Data) ---
    df_prophet = pd.DataFrame({'ds': series.index, 'y': series.values})
    model_prophet = Prophet(daily_seasonality=True).fit(df_prophet)
    
    # Forecast next 2 days
    future_2_days = pd.bdate_range(start=series.index[-1] + pd.Timedelta(days=1), periods=2)
    pred_prophet_2d = model_prophet.predict(pd.DataFrame({'ds': future_2_days}))['yhat'].values
    
    # --- MODEL 2: AUTO-ARIMA (Trained on Full Data) ---
    model_arima = auto_arima(series, seasonal=False, stepwise=True, trace=False)
    pred_arima_2d = model_arima.predict(n_periods=2)
    
    # Format array output
    if isinstance(pred_arima_2d, pd.Series):
        pred_arima_2d = pred_arima_2d.values
    
    # Ensemble Average for May 14 and May 15
    forecast_deliverables[stock_name] = {
        'Day 1 Forecast (May 14)': round((pred_prophet_2d[0] + pred_arima_2d[0]) / 2, 2),
        'Day 2 Forecast (May 15)': round((pred_prophet_2d[1] + pred_arima_2d[1]) / 2, 2)
    }
    print(f"✅ {stock_name} forecasting complete.")

df_forecasts = pd.DataFrame(forecast_deliverables).T
print("\n--- Task 3 Deliverable: Next 2-Day Forecasts (Ensemble Averages) ---")
print(df_forecasts)

13:08:20 - cmdstanpy - INFO - Chain [1] start processing


Executing Task 3: Model Training on FULL Data & Forecasting...



13:08:21 - cmdstanpy - INFO - Chain [1] done processing
13:08:22 - cmdstanpy - INFO - Chain [1] start processing


✅ RELIANCE forecasting complete.


13:08:22 - cmdstanpy - INFO - Chain [1] done processing
13:08:28 - cmdstanpy - INFO - Chain [1] start processing


...

✅ INFY forecasting complete.


13:08:30 - cmdstanpy - INFO - Chain [1] done processing
13:08:34 - cmdstanpy - INFO - Chain [1] start processing


✅ TMCV forecasting complete.


13:08:35 - cmdstanpy - INFO - Chain [1] done processing


✅ ITC forecasting complete.

--- Task 3 Deliverable: Next 2-Day Forecasts (Ensemble Averages) ---
          Day 1 Forecast (May 14)  Day 2 Forecast (May 15)
RELIANCE                  1411.08                  1411.20
HDFCBANK                   792.94                   793.81
INFY                      1145.13                  1144.41
TMCV                       414.89                   415.34
ITC                        306.47                   305.90


In [11]:
from statsmodels.tsa.seasonal import seasonal_decompose
import numpy as np
import pandas as pd

print("Executing Task 4: Volatility Estimation...\n")
volatility_results = []

for stock in tickers:
    stock_name = stock.split('.')[0]
    series = data_cleaned[stock]
    
    # Log Returns & Volatility
    log_returns = np.log(series / series.shift(1)).dropna()
    daily_vol = log_returns.std()
    annualized_vol = daily_vol * np.sqrt(252)
    
    # Trend Decomposition (using 30-day window)
    decomposition = seasonal_decompose(series, model='additive', period=30)
    current_trend = "Upward" if decomposition.trend.iloc[-15] > decomposition.trend.iloc[-45] else "Sideways/Consolidating"
    
    volatility_results.append({
        'Stock': stock_name,
        'Daily Volatility': round(daily_vol, 4),
        'Annualized Risk': f"{(annualized_vol*100):.2f}%",
        'Trend': current_trend
    })

df_vol = pd.DataFrame(volatility_results)
print("--- Task 4 Deliverable: Volatility & Trend Assessment ---")
print(df_vol.to_string(index=False))

Executing Task 4: Volatility Estimation...

--- Task 4 Deliverable: Volatility & Trend Assessment ---
   Stock  Daily Volatility Annualized Risk                  Trend
RELIANCE            0.0144          22.92% Sideways/Consolidating
HDFCBANK            0.0137          21.68% Sideways/Consolidating
    INFY            0.0155          24.68% Sideways/Consolidating
    TMCV            0.0082          12.97% Sideways/Consolidating
     ITC            0.0129          20.54% Sideways/Consolidating
